<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.__version__)

2.10.0+cu128


In [ ]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [ ]:
z = w*x+b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [ ]:
y_pred= torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [ ]:
import torch.nn.functional as F
loss = F.binary_cross_entropy(y_pred,y)
loss

tensor(6.7012, grad_fn=<BinaryCrossEntropyBackward0>)

In [ ]:
loss.backward()

In [ ]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


Training pipeline in pytorch


---



In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [ ]:
df.shape

(569, 31)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

array([[-0.81635369, -0.15368334, -0.81406988, ..., -0.44815544,
         0.24665917, -0.8804647 ],
       [ 2.05718603,  0.29180868,  1.97123294, ...,  1.75515701,
        -1.03519165, -0.52404586],
       [ 0.41726634,  3.33328303,  0.50483538, ...,  1.34410278,
         0.9782205 ,  2.03806148],
       ...,
       [-0.70741827, -0.60617263, -0.73051932, ..., -0.69138828,
        -1.93507683, -0.56956977],
       [-0.72213927, -0.51287587, -0.77399971, ..., -1.10955096,
        -0.28582684, -0.85714758],
       [ 0.7852914 , -0.10936738,  0.74355126, ...,  0.50963181,
        -0.15796546, -0.63341426]])

In [ ]:
y_train

,diagnosis
515,B
449,M
259,M
498,M
120,B
...,...
409,B
473,B
136,B
350,B


In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:

X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

torch.Size([455, 30])

In [ ]:
y_train_tensor.shape

torch.Size([455])

In [ ]:
import torch.nn as nn
class MySimple(nn.Module):
  def __init__(self,num_features):
     super().__init__()
     self.linear = nn.Linear(num_features, 1)
     self.sigmoid = nn.Sigmoid()


  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out


In [ ]:
learning_rate = 0.1
epochs = 25

In [ ]:
#automated loss_function
loss_function = nn.BCELoss()

Training pipeline

In [ ]:
# create model
model = MySimple(X_train_tensor.shape[1])

#defining optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

#training loop

for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor.float())

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1).float())



  # backward pass
  loss.backward()

  # parameters update
  """
  with torch.no_grad():
    model.linear.weight -= learning_rate * model.linear.weight.grad
    model.linear.bias -= learning_rate * model.linear.bias.grad


  # zero gradients
  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()
  """

  optimizer.step()
  optimizer.zero_grad()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.9477179646492004
Epoch: 2, Loss: 0.6653377413749695
Epoch: 3, Loss: 0.5181262493133545
Epoch: 4, Loss: 0.4346454441547394
Epoch: 5, Loss: 0.38128921389579773
Epoch: 6, Loss: 0.3440896272659302
Epoch: 7, Loss: 0.31650376319885254
Epoch: 8, Loss: 0.29509660601615906
Epoch: 9, Loss: 0.27790072560310364
Epoch: 10, Loss: 0.26371097564697266
Epoch: 11, Loss: 0.2517484426498413
Epoch: 12, Loss: 0.24148711562156677
Epoch: 13, Loss: 0.23255851864814758
Epoch: 14, Loss: 0.22469639778137207
Epoch: 15, Loss: 0.21770335733890533
Epoch: 16, Loss: 0.21142950654029846
Epoch: 17, Loss: 0.20575889945030212
Epoch: 18, Loss: 0.2006002813577652
Epoch: 19, Loss: 0.19588054716587067
Epoch: 20, Loss: 0.19154059886932373
Epoch: 21, Loss: 0.18753188848495483
Epoch: 22, Loss: 0.1838141679763794
Epoch: 23, Loss: 0.18035376071929932
Epoch: 24, Loss: 0.17712223529815674
Epoch: 25, Loss: 0.17409533262252808


In [ ]:
# create model
model = MySimple(X_train_tensor.shape[1])

#defining optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

#training loop

for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor.float())

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1).float())

  #clear gradiant
  optimizer.zero_grad()

  # backward pass
  loss.backward()

  # parameters update
  optimizer.step()
linear.bias.grad.zero_()


Epoch: 1, Loss: 0.8947765231132507
Epoch: 2, Loss: 0.6233835816383362
Epoch: 3, Loss: 0.49028533697128296
Epoch: 4, Loss: 0.4163694977760315
Epoch: 5, Loss: 0.3688025176525116
Epoch: 6, Loss: 0.33512941002845764
Epoch: 7, Loss: 0.30974486470222473
Epoch: 8, Loss: 0.2897469699382782
Epoch: 9, Loss: 0.273473858833313
Epoch: 10, Loss: 0.2599000632762909
Epoch: 11, Loss: 0.24835528433322906
Epoch: 12, Loss: 0.23838107287883759
Epoch: 13, Loss: 0.2296520173549652
Epoch: 14, Loss: 0.2219301015138626
Epoch: 15, Loss: 0.2150365114212036
Epoch: 16, Loss: 0.20883408188819885
Epoch: 17, Loss: 0.20321550965309143
Epoch: 18, Loss: 0.19809547066688538
Epoch: 19, Loss: 0.19340507686138153
Epoch: 20, Loss: 0.18908819556236267
Epoch: 21, Loss: 0.18509824573993683
Epoch: 22, Loss: 0.18139652907848358
Epoch: 23, Loss: 0.17795035243034363
Epoch: 24, Loss: 0.17473198473453522
Epoch: 25, Loss: 0.17171771824359894


In [ ]:
model.linear.weight

Parameter containing:
tensor([[ 2.2267e-01,  1.3065e-01,  1.4608e-01,  2.9306e-01,  1.4757e-01,
          2.1786e-01,  2.1980e-01,  1.1810e-01, -3.7265e-02,  8.8513e-02,
          2.7482e-02,  7.1789e-02,  2.1875e-01,  2.7622e-01,  7.9972e-02,
          2.4184e-02, -7.3507e-02,  7.8214e-02, -1.3245e-01, -1.6719e-01,
          3.1664e-01,  1.1895e-01,  3.5136e-01,  3.4207e-01,  2.3582e-02,
          1.1218e-01,  3.5303e-01,  1.5494e-01,  2.5514e-01,  2.1062e-04]],
       requires_grad=True)

In [ ]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor.float())
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5706371068954468


Advanced Neural Network

In [ ]:
from numpy._core.numeric import ones_like
#create model class
import torch
import torch.nn as nn

class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    #making container
    self.network = nn.Sequential(
         nn.Linear(num_features, 1),
         nn.ReLU(),
         nn.Linear(1,1), # Changed from nn.Linear(3,1) to nn.Linear(1,1)
         nn.Sigmoid())

  def forward(self, features):
     out = self.network(features)
     return out

In [ ]:
features = torch.rand(10,5)

model = Model(features.shape[1])
print(model(features))

tensor([[0.5489],
        [0.5449],
        [0.5489],
        [0.5489],
        [0.5489],
        [0.5489],
        [0.5449],
        [0.5489],
        [0.5479],
        [0.5485]], grad_fn=<SigmoidBackward0>)


In [ ]:
model.network[0].weight

Parameter containing:
tensor([[ 0.0608,  0.1553,  0.1461,  0.0850, -0.3542]], requires_grad=True)

In [ ]:
model.network[0].bias

Parameter containing:
tensor([-0.1127], requires_grad=True)

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

summary(model, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 1]                   6
│    └─ReLU: 2-2                         [10, 1]                   --
│    └─Linear: 2-3                       [10, 1]                   2
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 8
Trainable params: 8
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00